# 04 — ViT-Base/16 e Swin-T no Kaggle (ADR-0012)

Treina os **Vision Transformers** (objetivo central do TCC) com a **mesma metodologia** do baseline ResNet-50: mesmo `split_v1`, mesmo loop (PyTorch puro), hiperparâmetros V2 (LR diferenciado, dropout, augmentation forte), weighted CE, métricas balanceadas. Muda **só a arquitetura**.

Referências: [ADR-0012](https://github.com/ThiagoCB1900/TCC/blob/main/docs/decisions/0012-modelos-transformer-vit-base-swin-tiny.md) (modelos), [F-0017](https://github.com/ThiagoCB1900/TCC/blob/main/docs/findings/0017-baseline-v2-overfit-controlado-teto-resnet.md) (baseline fechado, teto ~0,59).

## Pré-requisitos (iguais ao notebook 03)
1. **+ Add Input** → `ninadaithal/imagesoasis`.
2. **Settings → Accelerator → GPU T4** + **Internet → On**.
3. Rodar com **Save Version → Save & Run All (Commit)** para imunidade a desconexão.

## Plano de treino
- ViT-Base/16 weighted (~2-3h) + Swin-T weighted (~2-3h). Cabe na cota de 30h/semana.
- Ablações sem peso ficam em células opcionais ao final.
- Comparação final ResNet(V2) vs ViT vs Swin + McNemar.

**Se der OOM** (ViT-Base é o maior, 86M params): reduza para `--batch-size 16` nas células de treino.

## 1. Setup (GPU, dataset, repo, deps) — idêntico ao notebook 03

In [ ]:
import os, glob, sys, torch
assert torch.cuda.is_available(), 'GPU nao detectada. Settings -> Accelerator -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

# Detecta o dataset em qualquer caminho sob /kaggle/input
hits = glob.glob('/kaggle/input/**/Non Demented', recursive=True)
assert hits, 'Adicione o dataset ninadaithal/imagesoasis em + Add Input.'
DATA_DIR = os.path.dirname(hits[0])
print('Data dir:', DATA_DIR)

# Clona/atualiza o repo
%cd /kaggle/working
if not os.path.isdir('/kaggle/working/TCC'):
    !git clone https://github.com/ThiagoCB1900/TCC.git
else:
    %cd /kaggle/working/TCC
    !git fetch --all && git reset --hard origin/main
%cd /kaggle/working/TCC
!git log --oneline -1

# Symlinks: Data/ -> dataset; experiments/runs -> /kaggle/working/runs
for link, target in [('/kaggle/working/TCC/Data', DATA_DIR),
                     ('/kaggle/working/TCC/experiments/runs', '/kaggle/working/runs')]:
    os.makedirs('/kaggle/working/runs', exist_ok=True)
    if os.path.islink(link) or os.path.exists(link):
        os.system(f'rm -rf {link}')
    os.symlink(target, link)

# Deps (PyTorch ja vem no Kaggle)
!pip install -q timm==1.0.11 monai==1.3.2 torchmetrics==1.4.3 tabulate==0.9.0 seaborn==0.13.2
if '/kaggle/working/TCC' not in sys.path:
    sys.path.insert(0, '/kaggle/working/TCC')
print('Setup OK')

## 2. Smoke test ViT (valida pipeline com a nova arquitetura, ~3 min)

Esperado: log mostra `Construindo modelo vit_base_16`, `LR diferenciado`, loss diminui, sem OOM.

In [ ]:
!python -m src.training.run \
    --model vit_base_16 --smoke --smoke-batches 20 \
    --batch-size 32 --batch-size-eval 64 --num-workers 2 \
    --run-name vit_smoke_kaggle

## 3. Treino ViT-Base/16 — V3 regularizado (ADR-0013) — ~3-4h

A V2 (rodada anterior) overfittou (best epoch 2, F-0018). V3 aplica regularização forte: `lr_backbone 5e-6`, `drop_path 0.2`, `weight_decay 0.2`, `warmup 3`. Esperado: best epoch mais tardio, val_loss sem explodir, balanced_acc no test acima de 0,521.

In [ ]:
# ViT-Base/16 — V3 (regularizacao forte, ADR-0013) — ~3-4h
!python -m src.training.run \
    --model vit_base_16 --epochs 20 \
    --lr-backbone 5e-6 --drop-path-rate 0.2 --weight-decay 0.2 --warmup-epochs 3 \
    --batch-size 32 --batch-size-eval 64 --num-workers 2 \
    --run-name vit_base_16_v3_class3_weighted_kaggle

## 4. Treino Swin-T — V3 regularizado (ADR-0013) — ~2h

Swin V2 já foi o melhor modelo (bal_acc 0,594, val pico 0,668) mesmo overfittando. V3 deve destravar parte desse ganho preso atrás do overfit.

In [ ]:
# Swin-T — V3 (regularizacao forte, ADR-0013) — ~2h
!python -m src.training.run \
    --model swin_tiny --epochs 20 \
    --lr-backbone 5e-6 --drop-path-rate 0.2 --weight-decay 0.2 --warmup-epochs 3 \
    --batch-size 32 --batch-size-eval 64 --num-workers 2 \
    --run-name swin_tiny_v3_class3_weighted_kaggle

## 5. Comparação ResNet(V2) vs ViT vs Swin + McNemar

Lê as runs weighted de cada arquitetura e compara. McNemar usa as predições por amostra (`test_predictions.npz`) — válido porque o test é o mesmo split determinístico.

In [ ]:
import json, numpy as np, pandas as pd
from pathlib import Path
from src.evaluation.metrics import mcnemar_test

runs_dir = Path('/kaggle/working/runs')

def latest(tag):
    cand = sorted([d for d in runs_dir.iterdir() if d.is_dir() and tag in d.name and 'smoke' not in d.name])
    return cand[-1] if cand else None

# Todas as variantes (as que existirem aparecem; as ausentes sao puladas)
targets = {
    'ResNet-50 V2':   'resnet50_v2_class3_weighted_kaggle',
    'ViT-Base V2':    'vit_base_16_class3_weighted_kaggle',
    'ViT-Base V3':    'vit_base_16_v3_class3_weighted_kaggle',
    'Swin-T V2':      'swin_tiny_class3_weighted_kaggle',
    'Swin-T V3':      'swin_tiny_v3_class3_weighted_kaggle',
}

rows, preds = [], {}
for label, tag in targets.items():
    run = latest(tag)
    if run is None:
        print(f'(sem run {label})'); continue
    f = json.loads((run / 'final_test_metrics.json').read_text(encoding='utf-8'))['test_metrics']
    rows.append({'modelo': label, 'accuracy': f['accuracy'], 'balanced_accuracy': f['balanced_accuracy'],
                 'macro_f1': f['macro_f1'], 'auc_macro': f['auc_macro'],
                 'F1_non': f['f1_per_class'].get('non_demented'),
                 'F1_very_mild': f['f1_per_class'].get('very_mild'),
                 'F1_mild_or_mod': f['f1_per_class'].get('mild_or_moderate')})
    npz = run / 'test_predictions.npz'
    if npz.exists():
        z = np.load(npz); preds[label] = (z['y_true'].copy(), z['y_pred'].copy()); z.close()

print(pd.DataFrame(rows).set_index('modelo').T.to_string())

print('\n=== McNemar (par-a-par, mesmo test set) ===')
labels = list(preds.keys())
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        a, b = labels[i], labels[j]
        mc = mcnemar_test(preds[a][0], preds[a][1], preds[b][1])
        sig = 'SIGNIFICATIVO' if mc['p_value'] < 0.05 else 'n.s.'
        print(f'  {a} vs {b}: b={mc["b"]} c={mc["c"]} p={mc["p_value"]:.4g} ({sig})')

## 6. Compactar artefatos para baixar e commitar

In [ ]:
import shutil
out = Path('/kaggle/working/runs_to_commit')
if out.exists(): shutil.rmtree(out)
out.mkdir(parents=True)
for run in runs_dir.iterdir():
    if not run.is_dir() or 'smoke' in run.name:
        continue
    tgt = out / run.name; tgt.mkdir(exist_ok=True)
    for fn in ['config.yaml','history.json','final_test_metrics.json','train.log','final_curves.png','test_predictions.npz']:
        if (run / fn).exists():
            shutil.copy(run / fn, tgt / fn)
shutil.make_archive('/kaggle/working/runs_to_commit', 'zip', out)
print('Pronto: /kaggle/working/runs_to_commit.zip — baixe e descompacte em experiments/runs/ local.')
!ls -la /kaggle/working/runs_to_commit.zip

## 7. (Opcional) Ablações sem peso — ViT e Swin

Rode só se houver cota/tempo. Para a comparação principal, as runs weighted bastam.

In [ ]:
# !python -m src.training.run --model vit_base_16 --epochs 20 --batch-size 32 --batch-size-eval 64 --num-workers 2 --no-class-weights --run-name vit_base_16_class3_noweight_kaggle
# !python -m src.training.run --model swin_tiny   --epochs 20 --batch-size 32 --batch-size-eval 64 --num-workers 2 --no-class-weights --run-name swin_tiny_class3_noweight_kaggle

## 8. Próximos passos

1. Baixar `runs_to_commit.zip`, descompactar em `experiments/runs/` local, commitar.
2. Me mandar a tabela comparativa + McNemar.
3. Se ViT/Swin platôarem ~0,60 como o ResNet → aplicar **2.5D** (ADR-0011) a todos.
4. Fase de interpretabilidade: Attention Rollout (ViT/Swin) + Grad-CAM (ResNet).